# 02 - Segmentation Experiments (Phase 2)

Reads the artifacts produced by `scripts/run_segmentation_audit.py` and reviews the
classical segmentation pipeline. This notebook does **not** tune anything and never
edits the raw dataset; all thresholds live in `backend/configs/segmentation.yaml` and
are changed only through the documented procedure in `docs/SEGMENTATION_NOTES.md`.

Run from the `notebooks/` directory after:

```powershell
python scripts/run_segmentation_audit.py
```

Split discipline: the 96 development images are the only ones that may inform a
threshold change; the 24 held-out test images are shown separately.

In [ ]:
from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED = REPO_ROOT / "data" / "processed" / "segmentation_audit"
DOCS = REPO_ROOT / "docs" / "segmentation_audit"

rows = pd.read_csv(PROCESSED / "seg-v2" / "segmentation_audit.csv")
summary = json.loads((DOCS / "audit_summary.json").read_text(encoding="utf-8"))
curated = json.loads((DOCS / "curated_manifest.json").read_text(encoding="utf-8"))

print("config:", summary["segmentation_config_version"], summary["segmentation_config_sha256"][:12])
print("quality config:", summary["quality_config_version"])
print("generated:", summary["generated_at"])
print("images:", len(rows))

## 1. Outcome counts

`clean` = no warnings, `uncertain` = warnings but analysable, `rejected` = analysis refused.
Development and held-out test are reported separately on purpose.

In [ ]:
table = pd.DataFrame(
    {
        "all 120": summary["overall"],
        "development (96)": summary["development"],
        "held-out test (24)": summary["held_out_test"],
    }
).loc[["images", "leaf_ok", "clean", "uncertain", "rejected",
       "full_frame_fallback", "multiple_leaves", "lesion_unreliable"]]
display(table)

## 2. Per class

In [ ]:
per_class = pd.DataFrame(summary["per_class"]).T
display(per_class[["images", "leaf_ok", "clean", "uncertain", "rejected", "full_frame_fallback"]])

## 3. Warnings raised

Warnings are what pushes a result towards `uncertain` in the Phase 4 API.

In [ ]:
warnings = pd.Series(summary["warning_histogram"]).sort_values()
ax = warnings.plot.barh(figsize=(7, 3), color="#0f766e")
ax.set_xlabel("images")
ax.set_title("Quality / segmentation warnings (seg-v2, 120 images)")
plt.tight_layout()
plt.show()

## 4. Lesion-area estimate

Lesion area is `lesion_pixels / leaf_pixels x 100`, the quantity the experimental
severity estimate (Phase 4) is built on. It is not agronomic severity.

In [ ]:
rows["lesion_area_percentage"].describe().to_frame().T

In [ ]:
ax = rows.boxplot(column="lesion_area_percentage", by="label", figsize=(8, 3.5), grid=False)
ax.set_ylabel("lesion area (%)")
ax.set_xlabel("")
ax.set_title("Lesion-area estimate by class (seg-v2)")
plt.suptitle("")
plt.tight_layout()
plt.show()

## 5. Curated visual evidence

The committed 4-up panels (input canvas, leaf mask, lesion mask, overlay) live in
`docs/segmentation_audit/`. They are the same files referenced by
`docs/SEGMENTATION_NOTES.md`.

In [ ]:
import matplotlib.image as mpimg

for item in curated["items"][:4]:
    image = mpimg.imread(DOCS / item["panel"])
    plt.figure(figsize=(14, 3.2))
    plt.imshow(image)
    plt.axis("off")
    plt.title(item["source"], fontsize=9)
    plt.show()

## 6. Known limitations

See `docs/SEGMENTATION_NOTES.md` section 6. In short: no segmentation ground truth,
thresholds calibrated on white-paper studio photos, a residual edge band on some
Bacterial leaf blight strips, and no real multi-leaf photograph in the dataset.